# CodeCrafters webinar evaluation: Colab analysis

This notebook is the execution entry point for the reproducible analysis pipeline. Run it in Google Colab. The raw CSV is read-only; derived tables and figures are written under `ANALYSIS/outputs/`.

## Colab setup

This notebook works with a Colab runtime or a VS Code kernel. It searches upward from the current working directory and also checks nested Colab/Drive folders for `ANALYSIS/data` or a root-level `data/` folder. If it cannot find the repository in Colab, it opens an upload dialog.

In [ ]:
%pip install -q "pandas>=2.1,<3" "numpy>=1.26,<3" "matplotlib>=3.8,<4" "seaborn>=0.13,<1"
print('Colab dependencies installed or already available.')

In [ ]:
from pathlib import Path
import sys
import zipfile

def is_project_root(candidate):
    analysis_dir = candidate / 'ANALYSIS'
    return (analysis_dir / 'src').is_dir() and (
        (analysis_dir / 'data').is_dir() or (candidate / 'data').is_dir()
    )

def find_project_root(search_roots):
    for search_root in search_roots:
        if not search_root.exists():
            continue
        upward_candidates = [search_root, *search_root.parents]
        for candidate in upward_candidates:
            if is_project_root(candidate):
                return candidate
        for analysis_dir in search_root.rglob('ANALYSIS'):
            candidate = analysis_dir.parent
            if is_project_root(candidate):
                return candidate
    return None

search_roots = [Path('/content'), Path('/content/drive/MyDrive'), Path.cwd()]
PROJECT_ROOT = find_project_root(search_roots)

if PROJECT_ROOT is None:
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError(
            'VS Code kernel could not find the repository from the current directory. '
            'Set PROJECT_ROOT to the manuscript folder containing ANALYSIS/src and data.'
        ) from exc
    print('No manuscript repository found. Upload a ZIP containing ANALYSIS/src and data.')
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
    if not zip_names:
        raise FileNotFoundError('Upload a ZIP of the manuscript repository, not only the notebook file.')
    for zip_name in zip_names:
        with zipfile.ZipFile(zip_name) as archive:
            archive.extractall('/content')
    PROJECT_ROOT = find_project_root([Path('/content'), Path.cwd()])

if PROJECT_ROOT is None:
    raise FileNotFoundError('Colab still cannot find ANALYSIS/data. Upload the repository ZIP or mount the correct Drive folder.')

sys.path.insert(0, str(PROJECT_ROOT))
data_dir = PROJECT_ROOT / 'ANALYSIS' / 'data'
if not data_dir.is_dir():
    data_dir = PROJECT_ROOT / 'data'
data_files = sorted(data_dir.glob('*.csv'))
if len(data_files) != 1:
    raise FileNotFoundError(f'Expected exactly one CSV in ANALYSIS/data; found {len(data_files)}.')
input_path = data_files[0]
print('Project root:', PROJECT_ROOT)
print('Data directory:', data_dir)
print('Input CSV:', input_path.name)
print('Python:', sys.version.split()[0])

In [ ]:
from ANALYSIS.src.pipeline.config import DEFAULT_OUTPUT
from ANALYSIS.src.pipeline.run_analysis import run_analysis

output_dir = PROJECT_ROOT / 'ANALYSIS' / 'outputs'
manifest = run_analysis(input_path=input_path, output_dir=output_dir)
print('Rows:', manifest['rows'])
print('Flagged scale-direction rows:', manifest['flagged_rows'])
print('Normalization scope:', manifest['normalization_scope'])

In [ ]:
import pandas as pd

tables = output_dir / 'tables'
likert = pd.read_csv(tables / 'likert_item_summary.csv')
summary = pd.read_csv(tables / 'overall_satisfaction_summary.csv')
themes = pd.read_csv(tables / 'thematic_counts.csv')
display(likert[['item_order', 'item_label', 'n', 'mean', 'sd', 'agree_pct']])
display(summary[['item_key', 'n', 'mean', 'sd', 'agree_pct']])
display(themes)

In [ ]:
from IPython.display import Image, display

figures = output_dir / 'figures'
display(Image(filename=str(figures / 'likert_item_means.png')))
display(Image(filename=str(figures / 'likert_distribution.png')))

In [ ]:
matrix = pd.read_csv(tables / 'mixed_methods_evidence_matrix.csv')
display(matrix)
print('Copy-ready quantitative table:')
print(likert[['item_order', 'item_label', 'n', 'mean', 'sd', 'agree_pct']].to_markdown(index=False))
print('Copy-ready summary table:')
print(summary[['item_key', 'n', 'mean', 'sd', 'agree_pct']].to_markdown(index=False))
print('Copy-ready thematic counts:')
print(themes.to_markdown(index=False))

## Interpretation boundary

The normalized Likert values describe respondents' post-event perceptions. Overall rating and satisfaction are retained as separate single-item summaries. The two flagged rows are corrected only for the 13 primary Likert items because their all-1 item pattern conflicts with their positive summary ratings. Open-ended code frequencies describe reported feedback and are not objective measures of learning. The design is nonexperimental and post-event, so these outputs do not establish causation, objective competence, or population-level generalizability.